# L19 · Explicación causal y equidad por rutas

**S19 · Lectura: Cap. 23 · 40–55 min**

## La pregunta

¿Importante para el modelo significa causal?

## Qué harás

Distinguir importancia asociativa, efecto y rutas causales.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una tabla de afirmaciones y de la evidencia que las respalda.

> **Idea clave:** Una explicación causal necesita un mecanismo y un supuesto, no solo un ranking.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = setup()

In [ ]:
# Una explicación necesita más que importancia
import matplotlib.pyplot as plt
labels = ['asociación', 'efecto', 'ruta', 'evidencia']
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(range(len(labels)), [0] * len(labels), "o-", lw=3)
for i, label in enumerate(labels):
    ax.text(i, 0.08, label, ha="center", va="bottom", fontsize=10)
ax.set_xlim(-.35, len(labels) - .65); ax.set_ylim(-.2, .3); ax.axis("off")
ax.set_title("Una explicación necesita más que importancia")
plt.tight_layout(); plt.show()

## 1. Calentamiento (5 min): ¿afirmación causal o asociativa?
Clasifica mentalmente cada frase y despliega la respuesta.

1. «Las mujeres de la muestra ganan, en promedio, 12 % menos.»
2. «Si cambiáramos el género de una candidata, su puntaje de contratación bajaría.»
3. «El código postal predice bien el impago.»
4. «Quitar el código postal del modelo elimina la discriminación racial.»
5. «Con la ocupación fija, el género sigue asociado al salario.»
6. «El efecto directo de A sobre Y es 0,15 con M fijo.»
7. «Las decisiones del algoritmo son iguales entre grupos (paridad demográfica).»
8. «La decisión de esta persona habría sido la misma en el mundo contrafactual donde su atributo protegido fuera otro.»

<details><summary>Ver respuestas</summary>

1. Asociativa (Nivel 1). 2. Causal (Nivel 3, contrafactual). 3. Asociativa. 4. Causal y, además, **falsa en general**: se quitan las variables sensibles pero quedan sus proxies. 5. Asociativa (condicional). 6. Causal (intervención sobre A con M fijo; Nivel 2). 7. Asociativa (criterio de Nivel 1). 8. Causal (equidad contrafactual, Nivel 3).


## 2. El modelo: A → M → Y y A → Y (Listado 23.1, §23.3)
$A$ atributo protegido (0/1), $M$ mediador (ocupación), $Y$ salario:
$$M=0{,}5A+\varepsilon_M,\qquad Y=0{,}15\,A+0{,}6\,M+\varepsilon_Y.$$
Verdad: efecto directo = **0,15**; indirecto = $0{,}5\times0{,}6=$ **0,30**; total = **0,45**.

### Completa tú
`descomponer(df)` debe devolver `(total, cde, indirecto)`:
- **total:** diferencia de medias de $Y$ entre $A=1$ y $A=0$;
- **cde:** coeficiente de $A$ en la regresión de $Y$ sobre $(A, M)$ (válido **sin** interacción $A\times M$);
- **indirecto:** $\text{total}-\text{cde}$.

In [ ]:
def simular(n=20000, directo=0.15, a_m=0.5, m_y=0.6, inter=0.0, seed=3, m2_en_y=False):
    r = np.random.default_rng(seed)
    A = r.binomial(1, .5, n); M = a_m * A + r.normal(size=n)
    M2 = 0.4 * A + r.normal(size=n)                                 # un SEGUNDO mediador (se usa en §5)
    Y = directo * A + m_y * M + inter * A * M + (0.5 * M2 if m2_en_y else 0) + r.normal(size=n)
    return pd.DataFrame({"A": A, "M": M, "M2": M2, "Y": Y})
df = simular()
print(df.head(3).round(3))

In [ ]:
# ✏️ Completa tú — descomponer
# Pista: ols(y, X) devuelve un dict; el coeficiente de A es coef[1] (coef[0] es la constante).
def descomponer(df):
    """Devuelve (total, cde, indirecto)."""
    raise NotImplementedError("completa descomponer")

In [ ]:
tot, cde, ind = descomponer(df)
print(f"efecto total {tot:.3f} · directo controlado {cde:.3f} · indirecto {ind:.3f}")
# EVALUATION-CELL
print("verdad del DGP: total 0.450 · directo 0.150 · indirecto 0.300")
lab_check("descomponer", lambda d: tuple(np.round(descomponer(d), 1)), [((df,), (0.4, 0.1, 0.3), 0.051)])

## 3. Ejercicio 23.6.2: efecto directo = 0
Sin discriminación directa (todo el efecto pasa por $M$), el CDE estimado debe converger a 0 y el indirecto debe quedarse en 0,30. Lo vemos al crecer $n$.

In [ ]:
filas = []
for n_ in (200, 1000, 5000, 20000, 80000):
    t, c, i = descomponer(simular(n_, directo=0.0, seed=11)); filas.append((n_, t, c, i))
show(pd.DataFrame(filas, columns=["n", "total", "directo (CDE)", "indirecto"]).set_index("n").round(3))

**Mira el resultado.** Con n pequeño el CDE no es exactamente 0. ¿Es evidencia de discriminación directa?

**Respuesta:**

No: es error de muestreo. Hay que mirar el **intervalo**: con n pequeño el IC del CDE contiene al 0. Reportar «CDE = 0,07» sin su IC convierte ruido en una acusación (o en una absolución). El mismo cuidado vale para el efecto indirecto.



## 4. Con interacción $A\times M$, «indirecto = total − CDE» deja de valer
Si $Y=0{,}15A+0{,}6M+0{,}4\,A\!\cdot\!M+\varepsilon$, el efecto directo **depende del nivel de $M$**: $\text{CDE}(m)=0{,}15+0{,}4m$. Ya no hay «el» CDE, y el natural (con $M$ en su valor *natural* bajo $A=0$) no coincide con el controlado para un $m$ cualquiera. Calculamos todo con la verdad estructural (conocemos el SCM) y con regresión con interacción.

In [ ]:
n = 200000; r = np.random.default_rng(5); A = r.binomial(1, .5, n); eM = r.normal(size=n); eY = r.normal(size=n)
M = lambda a: 0.5 * a + eM                                           # M(a) con el MISMO ruido
Yf = lambda a, m: 0.15 * a + 0.6 * m + 0.4 * a * m + eY
NDE = (Yf(1, M(0)) - Yf(0, M(0))).mean()                              # efecto directo NATURAL
NIE = (Yf(1, M(1)) - Yf(1, M(0))).mean()                              # efecto indirecto NATURAL
TOT = (Yf(1, M(1)) - Yf(0, M(0))).mean()
cde_m = lambda m: (Yf(1, m) - Yf(0, m)).mean()                        # efecto directo CONTROLADO en M = m
print(f"verdad: total {TOT:.3f} = NDE {NDE:.3f} + NIE {NIE:.3f}")
print("CDE(m) para m = -1, 0, 0.5, 1:", [round(float(cde_m(m)), 3) for m in (-1, 0, .5, 1)])
Mo = M(A); Yo = np.where(A == 1, Yf(1, Mo), Yf(0, Mo))               # lo que SE OBSERVA
b = ols(Yo, np.column_stack([A, Mo, A * Mo]))["coef"]
print(f"regresión con interacción: coef A = {b[1]:.3f}, coef M = {b[2]:.3f}, coef A×M = {b[3]:.3f}")
tot_o = Yo[A == 1].mean() - Yo[A == 0].mean(); cde_ing = ols(Yo, np.column_stack([A, Mo]))["coef"][1]
print(f"fórmula ingenua «total − CDE (sin interacción)»: {tot_o - cde_ing:.3f}  frente al NIE verdadero {NIE:.3f}")

**Mira el resultado.** ¿Qué número usarías como «efecto directo» en una auditoría si hay interacción, y qué debes declarar junto con él?

**Respuesta:**

Hay que elegir **qué** efecto directo se quiere: el controlado a un nivel $m$ concreto (y decir cuál) o el natural (con $M$ en el valor que tendría bajo $A=0$). El regresor sin interacción mezcla ambos y da un número que no corresponde a ninguno. Moraleja: el listado del libro es válido **solo sin interacción**; con ella, se declara el estimando antes de calcular.



## 5. Señal de alerta: un mediador omitido infla el efecto directo
Añadimos un segundo mediador $M_2$ ($A\to M_2\to Y$, efecto indirecto extra $0{,}4\times0{,}5=0{,}20$). Si no lo medimos y controlamos solo por $M$, esa ruta se contabiliza como «efecto directo».

In [ ]:
d2 = simular(40000, m2_en_y=True, seed=8)
tot2 = d2.loc[d2.A == 1, "Y"].mean() - d2.loc[d2.A == 0, "Y"].mean()
cde_sin = ols(d2["Y"].to_numpy(), d2[["A", "M"]].to_numpy())["coef"][1]                 # M2 omitido
cde_con = ols(d2["Y"].to_numpy(), d2[["A", "M", "M2"]].to_numpy())["coef"][1]            # M2 incluido
show(pd.Series({"total": tot2, "«directo» sin M2 (omitido)": cde_sin, "directo controlando M y M2": cde_con, "verdad": 0.15}).round(3).to_frame("estimación"))

**Mira el resultado.** El «efecto directo» sin M2 salió inflado. ¿Quién cree que discrimina y quién no discrimina en esta auditoría?

**Respuesta:**

Quien audita ve un directo mucho mayor que 0,15 y concluye discriminación directa que **no existe**: ese exceso es la ruta por $M_2$, que era una ruta indirecta (quizá legítima). El resultado depende de que el grafo de rutas esté completo; por eso el capítulo exige especificar el grafo **antes** y hacer análisis de sensibilidad.



## 6. La decisión normativa: ¿qué rutas son legítimas? (Completa tú)
Ningún test dice si «ocupación» es una ruta aceptable. El método solo **formaliza** la decisión: dos rutas con efectos conocidos y una declaración explícita de cuáles son legítimas.

Completa `declarar_rutas()`: para cada ruta, `legitima` (True/False) y una `justificacion` de al menos 25 caracteres que mencione **por qué** (p. ej., si el acceso a esa ocupación estuvo históricamente restringido por A). Después `efecto_discriminatorio(descomp, rutas)` suma las rutas declaradas **ilegítimas**.

In [ ]:
# ✏️ Completa tú — declarar_rutas
# Pista: No hay respuesta técnica única. Lo evaluable es que la decisión sea explícita y esté justificada.
def declarar_rutas():
    """Devuelve {ruta: {"legitima": bool, "justificacion": str}} para las dos rutas del modelo."""
    return {
        "A → Y (directa)": {"legitima": None, "justificacion": ""},
        "A → M → Y (vía ocupación)": {"legitima": None, "justificacion": ""},
    }

In [ ]:
rutas = declarar_rutas()
for k, v in rutas.items():
    assert isinstance(v["legitima"], bool), f"decide si «{k}» es legítima (True/False)"
    assert len(v["justificacion"].strip()) >= 25, f"justifica «{k}» (≥ 25 caracteres)"
def efecto_discriminatorio(descomp, rutas):
    total, cde, ind = descomp
    return (0 if rutas["A → Y (directa)"]["legitima"] else cde) + (0 if rutas["A → M → Y (vía ocupación)"]["legitima"] else ind)
d_base = descomponer(df)
alterno = {"A → Y (directa)": {"legitima": False, "justificacion": "x" * 30}, "A → M → Y (vía ocupación)": {"legitima": False, "justificacion": "x" * 30}}
show(pd.Series({"con TU declaración": efecto_discriminatorio(d_base, rutas), "si TODAS las rutas fueran ilegítimas": efecto_discriminatorio(d_base, alterno)}, name="efecto atribuible a discriminación").round(3).to_frame())

**Mira el resultado.** El mismo dato produce dos cifras de «discriminación» distintas. ¿Qué parte es estadística y qué parte normativa?

**Respuesta:**

Las **estimaciones** de cada ruta (0,15 y 0,30) son estadísticas, con su incertidumbre y sus supuestos de identificación. **Cuál se suma** es una decisión legal, ética y organizacional que ningún dato resuelve. Un informe honesto muestra las dos cosas por separado y no esconde la declaración dentro de la elección de controles.



## 7. Supuesto y bitácora

In [ ]:
fila_b = bitacora(
    pregunta="¿Qué parte del efecto de A sobre Y es atribuible a discriminación directa y qué parte pasa por una ruta declarada legítima?",
    metodo="Descomposición por regresión en un SCM lineal (CDE y efecto indirecto), con ruta A→M→Y declarada explícitamente",
    estimando="Efecto directo controlado de A sobre Y",
    numero_ic=f"directo {d_base[1]:.3f} (verdad 0,15) · indirecto {d_base[2]:.3f} (verdad 0,30) · total {d_base[0]:.3f}",
    supuesto=supuesto("no observable", "declarado y acotado", "la legitimidad de la ruta A→M→Y es una decisión normativa; además se supone no confusión M–Y y ausencia de mediadores omitidos"),
    decision="continuar", lab="L19")

---
## Qué queda para el libro
Quedan en el libro: `dowhy` (estimadores de mediación; no está instalado aquí), el caso de la brecha salarial con datos reales (Oaxaca-Blinder) y la equidad contrafactual de Kusner et al. (Nivel 3), que se discute pero no se simula.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*